In [3]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

if Path.cwd().name == "demos" and Path.cwd().parent.name == "notebooks":
    os.chdir("../..")

import requests
import pandas as pd
from dotenv import load_dotenv
from gasp import db

load_dotenv()
API_KEY = os.getenv("EIA_API_KEY")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
url = "https://api.eia.gov/v2/natural-gas/stor/wkly/data/"     # TODO: everything before the ? in your browser url

params = {
    "api_key": API_KEY,
    "frequency": "weekly",
    "data[0]": "value",
    "facets[series][]": "NW2_EPG0_SWO_R48_BCF",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "offset": 0,
    "length": 5000
}

r = requests.get(url, params=params)     # TODO: send the GET request with url and params
r.raise_for_status()                     # TODO: raise an error if the request failed

response = r.json()                      # TODO: turn the response into a python dict
rows = response["response"]["data"]                  # TODO: the list of rows inside response

len(rows)

876

In [5]:
print("Number of rows:", len(rows))
print("First row:", rows[0] if rows else "No data returned")

Number of rows: 876
First row: {'period': '2010-01-01', 'duoarea': 'R48', 'area-name': 'NA', 'product': 'EPG0', 'product-name': 'Natural Gas', 'process': 'SWO', 'process-name': 'Underground Storage - Working Gas', 'series': 'NW2_EPG0_SWO_R48_BCF', 'series-description': 'Weekly Lower 48 States Natural Gas Working Underground Storage (Billion Cubic Feet)', 'value': '3117', 'units': 'BCF'}


In [6]:
df = pd.DataFrame(rows)[["period", "value"]]
df["period"] = pd.to_datetime(df["period"])
df["value"] = pd.to_numeric(df["value"]).astype(float)
df["series"] = "NW2_EPG0_SWO_R48_BCF"
df = df[["period", "series", "value"]].sort_values("period").reset_index(drop=True)

df.head()

,period,series,value
0,2010-01-01,NW2_EPG0_SWO_R48_BCF,3117.0
1,2010-01-08,NW2_EPG0_SWO_R48_BCF,2850.0
2,2010-01-15,NW2_EPG0_SWO_R48_BCF,2607.0
3,2010-01-22,NW2_EPG0_SWO_R48_BCF,2521.0
4,2010-01-29,NW2_EPG0_SWO_R48_BCF,2406.0


In [7]:
db.init_db()

first_write = db.write_series(df)
second_write = db.write_series(df)

print("First write:", first_write)
print("Second write:", second_write)

First write: 876
Second write: 0


In [8]:
# Create a copy of the original data
revision_df = df.iloc[[0]].copy()

# Change one value to simulate an EIA correction
revision_df.loc[revision_df.index[0], "value"] += 100.0

# Use a separate test series so existing data isn't affected
revision_df["series"] = "REVISION_DEMO"

original_df = df.iloc[[0]].copy()
original_df["series"] = "REVISION_DEMO"

print("Original value:", original_df["value"].iloc[0])
print("Revised value:", revision_df["value"].iloc[0])

Original value: 3117.0
Revised value: 3217.0


In [9]:
# Save original measurement as retrieved on January 10
original_count = db.write_series(
    original_df,
    retrieved_at="2024-01-10"
)

# Save revised measurement as retrieved on January 20
revision_count = db.write_series(
    revision_df,
    retrieved_at="2024-01-20"
)

print("Original rows added:", original_count)
print("Revision rows added:", revision_count)

Original rows added: 1
Revision rows added: 1


In [10]:
before = db.read_series(
    "REVISION_DEMO",
    as_of="2024-01-15"
)

after = db.read_series(
    "REVISION_DEMO",
    as_of="2024-01-25"
)

print("Before revision:")
display(before)

print("After revision:")
display(after)

Before revision:


,period,series,value
0,2010-01-01,REVISION_DEMO,3117.0


After revision:


,period,series,value
0,2010-01-01,REVISION_DEMO,3217.0


In [11]:
# Test what happens when an older retrieval arrives later
late_old = original_df.copy()
late_old["series"] = "OUT_OF_ORDER_TEST"

late_new = revision_df.copy()
late_new["series"] = "OUT_OF_ORDER_TEST"

print("Newer revision:", db.write_series(
    late_new, retrieved_at="2024-01-20"
))

print("Older observation:", db.write_series(
    late_old, retrieved_at="2024-01-10"
))

print("Before January 20:")
display(db.read_series("OUT_OF_ORDER_TEST", as_of="2024-01-15"))

print("Latest:")
display(db.read_series("OUT_OF_ORDER_TEST"))

Newer revision: 1
Older observation: 1
Before January 20:


,period,series,value
0,2010-01-01,OUT_OF_ORDER_TEST,3117.0


Latest:


,period,series,value
0,2010-01-01,OUT_OF_ORDER_TEST,3217.0


In [13]:
# Check whether repeating the older observation adds a duplicate
duplicate_count = db.write_series(
    late_old,
    retrieved_at="2024-01-10"
)

print("Duplicate rows added:", duplicate_count)

Duplicate rows added: 0


In [14]:
# Use a new series to keep this test separate
same_value_df = original_df.copy()
same_value_df["series"] = "SAME_VALUE_TEST"

# First retrieval
first = db.write_series(
    same_value_df,
    retrieved_at="2024-01-10"
)

# Same value, but retrieved five days later
second = db.write_series(
    same_value_df,
    retrieved_at="2024-01-15"
)

print("First retrieval:", first)
print("Second retrieval:", second)

First retrieval: 1
Second retrieval: 1


In [15]:
# Create a fresh test series
test_df = original_df.copy()
test_df["series"] = "DUPLICATE_TEST_2"

# Save the first value
first = db.write_series(
    test_df,
    retrieved_at="2024-01-10"
)

# Save the exact same value on a different date
second = db.write_series(
    test_df,
    retrieved_at="2024-01-15"
)

print("First retrieval:", first)
print("Second retrieval:", second)

First retrieval: 1
Second retrieval: 0


In [16]:
# Create a new series for the revision test
first_version = original_df.copy()
first_version["series"] = "REVERSAL_TEST"

second_version = first_version.copy()
second_version["value"] += 100.0

third_version = first_version.copy()

# Save three historical versions
print("First:", db.write_series(
    first_version, retrieved_at="2024-01-10"
))

print("Second:", db.write_series(
    second_version, retrieved_at="2024-01-20"
))

print("Third:", db.write_series(
    third_version, retrieved_at="2024-01-30"
))

print("Latest value:")
display(db.read_series("REVERSAL_TEST"))

First: 1
Second: 1
Third: 0
Latest value:


,period,series,value
0,2010-01-01,REVERSAL_TEST,3217.0


In [17]:
# Create three versions of the same measurement
version1 = original_df.copy()
version1["series"] = "REVERSAL_TEST_2"

version2 = version1.copy()
version2["value"] += 100.0

version3 = version1.copy()

# Write all three versions
print("First:", db.write_series(
    version1, retrieved_at="2024-01-10"
))

print("Second:", db.write_series(
    version2, retrieved_at="2024-01-20"
))

print("Third:", db.write_series(
    version3, retrieved_at="2024-01-30"
))

# Check the historical results
print("Before first revision:")
display(db.read_series("REVERSAL_TEST_2", as_of="2024-01-15"))

print("After first revision:")
display(db.read_series("REVERSAL_TEST_2", as_of="2024-01-25"))

print("After second revision:")
display(db.read_series("REVERSAL_TEST_2", as_of="2024-02-01"))

First: 1
Second: 1
Third: 1
Before first revision:


,period,series,value
0,2010-01-01,REVERSAL_TEST_2,3117.0


After first revision:


,period,series,value
0,2010-01-01,REVERSAL_TEST_2,3217.0


After second revision:


,period,series,value
0,2010-01-01,REVERSAL_TEST_2,3117.0


In [18]:
# Use a fresh series
older = original_df.copy()
older["series"] = "ORDER_TEST_3"

newer = older.copy()
newer["value"] += 100.0

# Insert the newer observation FIRST
print("Newer write:", db.write_series(
    newer, retrieved_at="2024-01-20"
))

# Insert the older observation SECOND
print("Older write:", db.write_series(
    older, retrieved_at="2024-01-10"
))

# Check historical results
print("As of January 15:")
display(db.read_series("ORDER_TEST_3", as_of="2024-01-15"))

print("As of January 25:")
display(db.read_series("ORDER_TEST_3", as_of="2024-01-25"))

Newer write: 1
Older write: 1
As of January 15:


,period,series,value
0,2010-01-01,ORDER_TEST_3,3117.0


As of January 25:


,period,series,value
0,2010-01-01,ORDER_TEST_3,3217.0


In [19]:
# Fresh series for the final test
original = original_df.copy()
original["series"] = "FINAL_CHECK_4"

revised = original.copy()
revised["value"] += 100.0

# Store the original and a later revision
print("Original:", db.write_series(
    original, retrieved_at="2024-01-10"
))

print("Revision:", db.write_series(
    revised, retrieved_at="2024-01-30"
))

# Insert a repeated original value between them
print("Repeated original:", db.write_series(
    original, retrieved_at="2024-01-20"
))

# Check that the revision remains correct
print("As of January 25:")
display(db.read_series("FINAL_CHECK_4", as_of="2024-01-25"))

print("As of February 1:")
display(db.read_series("FINAL_CHECK_4", as_of="2024-02-01"))

Original: 1
Revision: 1
Repeated original: 0
As of January 25:


,period,series,value
0,2010-01-01,FINAL_CHECK_4,3117.0


As of February 1:


,period,series,value
0,2010-01-01,FINAL_CHECK_4,3217.0


In [2]:
from pathlib import Path

print("Notebook working directory:", Path.cwd())
print("Database location:", Path("data/gasp.db").resolve())

Notebook working directory: c:\Users\adeni\Documents\fall2026-gasp
Database location: C:\Users\adeni\Documents\fall2026-gasp\data\gasp.db


In [7]:
# Check the database in the correct location
db.init_db()

# Write one test observation
location_test = df.iloc[[0]].copy()
location_test["series"] = "LOCATION_CHECK"

rows_added = db.write_series(location_test)

print("Rows added:", rows_added)
print("Data retrieved:")
display(db.read_series("LOCATION_CHECK"))

Rows added: 1
Data retrieved:


,period,series,value
0,2010-01-01,LOCATION_CHECK,3117.0
